In [1]:
import json
from pathlib import Path
import navis
import numpy
import pandas
import random

In [2]:
from acanalysis.acalignment.keypoints import read_keypoints
from acanalysis.acalignment.utils.swc_utils import read_neurons_from_file, current_id_func

In [3]:
from acanalysis.acalignment.utils.kevin_utils import get_tar_path, get_swc_path, get_roi_label, get_section_label

In [12]:
align_json = "align_S32_S33_roi2_mip1_240403.json"
with open(align_json) as f:
    alignjs = json.load(f)
mdpath = Path(alignjs["output_path"])
#outpath = mdpath
outpath = Path("./materialized/roi2")
if not outpath.exists():
    outpath.mkdir(parents=True)

In [5]:
def transform_neurons(skels,shift_xyz=[0,0,0],scale_xyz=[1,1,1]):
    M = numpy.diag([scale_xyz[0],scale_xyz[1],scale_xyz[2],1])
    if shift_xyz:
        M[:3,3] = numpy.array(shift_xyz)
    tr = navis.transforms.AffineTransform(M)
    return navis.xform(skels, tr)

def swap_axons_xyz(axons,swap_xyz):
    for a in axons:
        a.nodes[["x","y","z"]] = a.nodes[swap_xyz]
        print("[x,y,z] axes permuted as " + str(swap_xyz))

In [6]:
section_keypts_df = []
for s in alignjs["sections"]:
    kpfile = mdpath / Path(get_section_label(s) + "_inliers.json")
    keypts = read_keypoints(kpfile)
    df = pandas.DataFrame(keypts)
    get_id = lambda row: row["name"].split("_")[-1]
    get_pos = lambda row: row["name"].split("_")[-2]
    df["id"] = df.apply(get_id,axis=1)
    df["pos"] = df.apply(get_pos,axis=1)
    section_keypts_df.append(df)

In [7]:
section_neurons = [navis.NeuronList([]) for _ in alignjs["sections"]]
for i_s,(s,df) in enumerate(zip(alignjs["sections"],section_keypts_df)):
    is_tar = s["swc_kwargs"]["is_tar"]
    for i_tile,tile in enumerate(s["tiles"]):
        filepath = get_tar_path(s,i_tile) if is_tar else get_swc_path(s,i_tile)
        tilename = get_roi_label(s,i_tile) + "_"
        id_list = df.loc[df["pos"]==tile]["id"].tolist()
        print(len(id_list))
        swap_xyz = s["swc_kwargs"]["swap_xyz"]
        neurons = read_neurons_from_file(filepath,prefix=tilename,is_tar=is_tar,swap_xyz=swap_xyz,id_list=id_list)
        print(neurons.shape)
        section_neurons[i_s] += transform_neurons(neurons,shift_xyz=s["position_offsets"][i_tile])

23
(23,)


Xforming:   0%|          | 0/23 [00:00<?, ?it/s]

29
(29,)


Xforming:   0%|          | 0/29 [00:00<?, ?it/s]

28
(28,)


...

Xforming:   0%|          | 0/36 [00:00<?, ?it/s]

17
(17,)


Xforming:   0%|          | 0/17 [00:00<?, ?it/s]

19
(19,)


Xforming:   0%|          | 0/19 [00:00<?, ?it/s]

In [8]:
def flatten_neuron(neuron,df):
    kp = df.loc[(df["pos"]==neuron.name.split("_")[-2])&(df["id"]==neuron.id)]
    trM = numpy.eye(4)
    trM[0,3] = -kp["location"].to_numpy()[0][0]
    return navis.xform(neuron,navis.transforms.AffineTransform(trM))

In [9]:
flattened = []
for neurons,df in zip(section_neurons,section_keypts_df):
    func = lambda neuron: flatten_neuron(neuron,df)
    flattened.append(neurons.apply(func))

Apply <lambda>:   0%|          | 0/319 [00:00<?, ?it/s]

Apply <lambda>:   0%|          | 0/319 [00:00<?, ?it/s]

In [10]:
tformed = flattened
best_model_M = numpy.load(mdpath / Path("model.npy"))
rigidM_2D = best_model_M
rigidM = numpy.eye(4)
rigidM[1:3,1:3] = rigidM_2D[:2,:2]
rigidM[1:3:,3] = rigidM_2D[:2,2]
sb_skeleton_rigid_tform = rigidM
aff = navis.transforms.AffineTransform(sb_skeleton_rigid_tform)
tformed[1] = navis.xform(flattened[1], aff)

Xforming:   0%|          | 0/319 [00:00<?, ?it/s]

In [ ]:
sa,sb = tformed
#n_disp = 200
#k = random.sample(range(sa.shape[0]),n_disp)
colors = ['r' for n in sa] + ['b' for n in sb]
navis.plot3d(sa+sb,colors=colors,backend="plotly")

In [11]:
filenames = [get_section_label(s) + "_inliers_affine" for s in alignjs["sections"]]
for neurons,fn in zip(tformed,filenames):
    navis.write_swc(neurons,outpath / Path(fn + ".zip"))

Writing:   0%|          | 0/126 [00:00<?, ?it/s]

Writing:   0%|          | 0/126 [00:00<?, ?it/s]

In [12]:
from acanalysis.acalignment.utils.olga_utils import swc_multi_to_single

In [13]:
for fn in filenames:
    swc_multi_to_single(outpath / Path(fn + ".zip"),outpath,fn + ".swc",sort=True)

number of axons 126
(2549, 7)
number of axons 126
(2247, 7)


In [14]:
tiffmip = 1
mipdiff = [s["swc_mip"] - tiffmip for s in alignjs["sections"]]
print(mipdiff)
swcshifts = [[0,0,260],[0,0,260]]

[-1, -1]


In [15]:
trees = []
for fn in filenames:
    trees.append(navis.read_swc(outpath / Path(fn + ".swc")))
swap_axons_xyz(trees,["z","y","x"])
trees = [transform_neurons(tree,scale_xyz=[2**m,2**m,2**m]) for tree,m in zip(trees,mipdiff)]
trees = [transform_neurons(tree,shift_xyz=shift) for tree,shift in zip(trees,swcshifts)]
for tree,fn in zip(trees,filenames):
    navis.write_swc(tree,outpath / Path(fn + "_aivia.swc"))

[x,y,z] axes permuted as ['z', 'y', 'x']
[x,y,z] axes permuted as ['z', 'y', 'x']
